# Lab 5 - Streaming Agent Sessions

**Estimated time:** 15 min  ·  **Estimated cost:** $0.01

> **Cost note:** the meter prints a list-price estimate from `session.usage` plus Managed Agents active runtime. It is not an authoritative invoice. Check Claude Console billing for final cost.

![Architecture](assets/architecture.svg)

## Goal
Build a tiny streaming console: chat with one persistent session, watch the agent's text and tool use stream in live, and see a running cost ticker after every turn. You will learn to read the streaming event loop, surface tool calls in band, and turn `session.usage` into a dollar figure on your own UI.

## Prerequisites
- the shared uv kernel `Managed Agents Labs (.venv)`
- An API key you can paste into the setup cell below. The notebook stores it as `ANTHROPIC_API_KEY` for this kernel session only.

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()  # Loads from .env file

ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")
GOOGLE_DOCS_VAULT_ID = os.environ.get("GOOGLE_DOCS_VAULT_ID")
MODEL = os.environ.get("MODEL")
GOOGLE_DOCS_MCP_URL = os.environ.get("GOOGLE_DOCS_MCP_URL")


In [3]:
# Verify that this notebook is using the uv-managed lab kernel.
import sys
from pathlib import Path

EXPECTED_KERNEL = "Managed Agents Labs (.venv)"
python_exe = Path(sys.executable)
python_prefix = Path(sys.prefix)
print(python_exe)
print(python_prefix)

if ".venv" not in python_exe.parts and ".venv" not in python_prefix.parts:
    raise RuntimeError(
        f"Select the Jupyter kernel {EXPECTED_KERNEL!r} before running this notebook. "
        "From the repo root, run ./setup_uv.sh once to create and register it."
    )

print("Using uv environment:", python_prefix)

/Users/madhavathavale/workspace/MacMiniM2/AI-Code/claude-agents/claude-managed-agents-labs/.venv/bin/python
/Users/madhavathavale/workspace/MacMiniM2/AI-Code/claude-agents/claude-managed-agents-labs/.venv
Using uv environment: /Users/madhavathavale/workspace/MacMiniM2/AI-Code/claude-agents/claude-managed-agents-labs/.venv


In [4]:
import getpass
import os

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ")

print("ANTHROPIC_API_KEY configured for this notebook session.")

ANTHROPIC_API_KEY configured for this notebook session.


In [5]:
import os, sys
from anthropic import Anthropic

# Udemy Labs note: the previous cell configures ANTHROPIC_API_KEY for this session.
assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY first."

BETAS = ["managed-agents-2026-04-01"]
MODEL = os.environ.get("MODEL", "claude-haiku-4-5-20251001")  # course default; swap as models update
client = Anthropic()
print("SDK ready, model:", MODEL)

SDK ready, model: claude-haiku-4-5-20251001


### Step 1 - Create the agent, environment, and session
Create them once; the whole session console talks to this single session.

In [6]:
agent = client.beta.agents.create(
    name="Streaming Console",
    model=MODEL,
    system="You are a concise console assistant. Use the bash tool to inspect the workspace when asked. Keep replies short.",
    tools=[{"type": "agent_toolset_20260401"}],
    betas=BETAS,
)
env = client.beta.environments.create(
    name="streaming-agent-sessions",
    config={"type": "cloud", "networking": {"type": "unrestricted"}},
    betas=BETAS,
)
session = client.beta.sessions.create(
    agent=agent.id, environment_id=env.id, title="Lab 5 streaming agent sessions", betas=BETAS,
)
print("session.id =", session.id)

session.id = sesn_01SPX3FXuZcdvr1r5pJJ8sUc


### Step 2 - A cost helper
Re-fetch the session after each idle and print the shared list-price estimate from cumulative `session.usage` plus Managed Agents active runtime.

In [7]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "shared"))
from cost_meter import print_sessions_cost


def show_cost(session_id):
    print_sessions_cost(client, [session_id], MODEL, betas=BETAS)


### Step 3 - Send one turn and stream it
In a notebook we send a turn and consume the stream in a function (no blocking `input()` loop). Run this cell, then change `prompt` and run again to continue the same session.

In [8]:
def turn(prompt):
    with client.beta.sessions.events.stream(session.id, betas=BETAS) as stream:
        client.beta.sessions.events.send(
            session.id,
            events=[{"type": "user.message", "content": [{"type": "text", "text": prompt}]}],
            betas=BETAS,
        )
        sys.stdout.write("agent> "); sys.stdout.flush()
        for event in stream:
            if event.type == "agent.message":
                for block in event.content:
                    if block.type == "text":
                        sys.stdout.write(block.text); sys.stdout.flush()
            elif event.type == "agent.tool_use":
                sys.stdout.write(f"\n  [tool: {event.name}]\n  "); sys.stdout.flush()
            elif event.type == "session.status_idle":
                show_cost(session.id); break

turn("What's in /workspace?")

agent> 
  [tool: bash]
  The `/workspace` directory is empty. If you have files to work with, they might be in `/mnt/session/uploads` or I can help you create new files there.
Estimated lab cost (USD, list-price estimate):
  session sesn_01SPX3FXuZcdvr1r5pJJ8sUc: $0.0076 (16 in / 131 out; 4955 cache read / 5121 cache write; 2.5s runtime)
  total across 1 session(s): $0.0076
  total tokens: 16 input / 131 output; 4955 cache read / 5121 cache write
  total active runtime: 2.5s
  total line items: input $0.0000, output $0.0007, cache write $0.0064, cache read $0.0005, runtime $0.0001, web search $0.0000
  Note: this is not an invoice. Check Claude Console billing for authoritative cost.


In [9]:
# Run another turn on the same session - the cost total climbs.
turn("Read /etc/os-release and summarize it in one line.")

agent> 
  [tool: read]
  **Ubuntu 24.04.5 LTS (Noble Numbat)** – a Debian-based Linux distribution.
Estimated lab cost (USD, list-price estimate):
  session sesn_01SPX3FXuZcdvr1r5pJJ8sUc: $0.0099 (32 in / 254 out; 15052 cache read / 5604 cache write; 5.1s runtime)
  total across 1 session(s): $0.0099
  total tokens: 32 input / 254 output; 15052 cache read / 5604 cache write
  total active runtime: 5.1s
  total line items: input $0.0000, output $0.0013, cache write $0.0070, cache read $0.0015, runtime $0.0001, web search $0.0000
  Note: this is not an invoice. Check Claude Console billing for authoritative cost.


In [10]:
# Run another turn on the same session and check the previous messages
turn("What were my questions that I've already asked you?")

agent> You've asked me three questions so far:

1. "What's in /workspace?" 
2. "Read /etc/os-release and summarize it in one line."
3. "What were my questions that I've already asked you?" (current)
Estimated lab cost (USD, list-price estimate):
  session sesn_01SPX3FXuZcdvr1r5pJJ8sUc: $0.0118 (42 in / 436 out; 20194 cache read / 5906 cache write; 6.9s runtime)
  total across 1 session(s): $0.0118
  total tokens: 42 input / 436 output; 20194 cache read / 5906 cache write
  total active runtime: 6.9s
  total line items: input $0.0000, output $0.0022, cache write $0.0074, cache read $0.0020, runtime $0.0002, web search $0.0000
  Note: this is not an invoice. Check Claude Console billing for authoritative cost.


### Cost estimate
Re-fetch the session id(s), print one list-price estimate per session, then print the total across all session ids. The estimate uses cumulative token usage plus Managed Agents active runtime; treat it as a course meter, not an invoice.


In [11]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "shared"))
from cost_meter import print_sessions_cost

print_sessions_cost(client, [session.id], MODEL, betas=BETAS)



Estimated lab cost (USD, list-price estimate):
  session sesn_01SPX3FXuZcdvr1r5pJJ8sUc: $0.0118 (42 in / 436 out; 20194 cache read / 5906 cache write; 6.9s runtime)
  total across 1 session(s): $0.0118
  total tokens: 42 input / 436 output; 20194 cache read / 5906 cache write
  total active runtime: 6.9s
  total line items: input $0.0000, output $0.0022, cache write $0.0074, cache read $0.0020, runtime $0.0002, web search $0.0000
  Note: this is not an invoice. Check Claude Console billing for authoritative cost.


## Expected output
- A `session.id` line.
- Agent text streaming after `agent> `, with a `[tool: bash]` marker each time it reaches for a tool.
- An `Estimated lab cost` block after each turn, climbing as usage and active runtime accumulate.

## Troubleshooting
- **Cost always $0.00 / `usage` is None** → read usage only after `session.status_idle`, and re-fetch with `client.beta.sessions.retrieve(...)` (the create-time object is stale).
- **Stream hangs** → SSE is one long connection; a dropped network closes it. Re-run the `turn(...)` cell to open a fresh stream. To catch up an in-flight turn, `client.beta.sessions.events.list(session.id, betas=BETAS)`.
- **The number looks wrong** → it is a list-price estimate, not an invoice. Source final numbers from Claude Console billing. Cache counters come from `session.usage.cache_read_input_tokens` and `session.usage.cache_creation_input_tokens`.

## Bonus (optional): drive this lab with Claude Code

Not required - the notebook above is the whole lab. If you want to try **agentic engineering**, open this folder in Claude Code and paste:

> "Write a small streaming console for the Claude Managed Agents beta (`betas=['managed-agents-2026-04-01']`, all calls on `client.beta.*`). Create an agent on `claude-haiku-4-5-20251001` with `{'type': 'agent_toolset_20260401'}`, a plain cloud environment, and one session. Send a turn, open `client.beta.sessions.events.stream(...)`, print `agent.message` text live and every `agent.tool_use` as `[tool: <name>]`, and on `session.status_idle` print a running cost from `session.usage`."

Then ask it to reuse `../shared/cost_meter.py` for the running estimate. Compare what Claude Code writes to the cells above.

## Stretch
- Color the output with `rich` (tool markers cyan, cost line green).
- Keep a `collections.Counter` of tool names and print `bash x3, edit x1` alongside the cost.
- Persist `session.id` and reattach on the next run instead of creating a new session.